# PPMI — ConShift anchor / shift tables

Loads **`results/anchors_ppmi/{source}_to_{target}_shifts.tsv`** from `code/anchors_conshift.py --method ppmi`.

*(Optional legacy: `results/ppmi_{a}_{b}.tsv` from `compute_distances.py` — no header — is also supported.)*

**Top-*N* tables:** one table per pair, columns **`word`**, **`shift`** (distance), **`count_a`**, **`count_b`**. Counts are **Word2Vec training token counts** (same proxy as the Word2Vec notebook).

**Pairs:** any `*_to_*` pairs found in `results/anchors_ppmi/` (including `arxiv->reddit`, `yelp->reddit`, `ciao->reddit` if present).

In [1]:
from pathlib import Path
import pandas as pd

TOP_N = 10
CORPORA = ["arxiv", "yelp", "ciao", "reddit"]

candidates_repo = [
    Path(".").resolve(),
    Path("..").resolve(),
    Path("../..").resolve(),
]
REPO_ROOT = next(
    (p for p in candidates_repo if (p / "results").is_dir() and (p / "embeddings" / "word2vec").is_dir()),
    Path("..").resolve(),
)
RESULTS_ROOT = REPO_ROOT / "results"
ANCHORS_DIR = RESULTS_ROOT / "anchors_ppmi"

shift_files = []
for path in sorted(ANCHORS_DIR.glob("*_shifts.tsv")):
    pair_slug = path.stem.replace("_shifts", "")
    a, b = pair_slug.split("_to_", 1)
    shift_files.append((path, a, b))

if not shift_files:
    for a, b in [("arxiv", "yelp"), ("arxiv", "ciao"), ("yelp", "ciao")]:
        p = RESULTS_ROOT / f"ppmi_{a}_{b}.tsv"
        if p.is_file():
            shift_files.append((p, a, b))

print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"RESULTS_ROOT  = {RESULTS_ROOT}")
print(f"ANCHORS_DIR   = {ANCHORS_DIR}")
print("Loaded:", [t[0].name for t in shift_files])

Missing (skip): ppmi_arxiv_yelp.tsv
Missing (skip): ppmi_arxiv_ciao.tsv
Missing (skip): ppmi_yelp_ciao.tsv
REPO_ROOT    = C:\work_study\github\UofT\CSC2611\synchronic-semantic-variation
RESULTS_ROOT = C:\work_study\github\UofT\CSC2611\synchronic-semantic-variation\results
Loaded: []


In [2]:
def parse_pair(pair_name: str):
    a, b = pair_name.split("_to_", 1)
    return a, b


_wv_cache: dict = {}


def wv_word_count(wv, w: str) -> int:
    if hasattr(wv, "get_vecattr"):
        try:
            c = wv.get_vecattr(w, "count")
            if c is not None:
                return int(c)
        except (KeyError, ValueError, TypeError):
            pass
    vocab = getattr(wv, "vocab", None)
    if vocab is not None and w in vocab:
        return int(vocab[w].count)
    return 0


def get_wv(corpus: str):
    if corpus not in _wv_cache:
        from gensim.models import Word2Vec

        path = REPO_ROOT / "embeddings" / "word2vec" / f"{corpus}.model"
        if not path.is_file():
            raise FileNotFoundError(f"Missing Word2Vec model: {path}")
        m = Word2Vec.load(str(path))
        _wv_cache[corpus] = m.wv
    return _wv_cache[corpus]


def attach_pair_counts(df: pd.DataFrame, src: str, tgt: str) -> pd.DataFrame:
    cs, ct = f"count_{src}", f"count_{tgt}"
    wv_a, wv_b = get_wv(src), get_wv(tgt)
    out = df.copy()
    out[cs] = [wv_word_count(wv_a, w) for w in out["word"]]
    out[ct] = [wv_word_count(wv_b, w) for w in out["word"]]
    return out


def load_ppmi_tsv(path: Path, src: str, tgt: str) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        first = f.readline()
    if first.lower().startswith("word\t"):
        df = pd.read_csv(path, sep="\t")
    else:
        df = pd.read_csv(path, sep="\t", header=None, names=["word", "shift"])
    pair_name = f"{src}_to_{tgt}"
    df["pair"] = pair_name
    cs, ct = f"count_{src}", f"count_{tgt}"
    if cs in df.columns and ct in df.columns:
        return df
    return attach_pair_counts(df, src, tgt)


def pairwise_top_n(
    df: pd.DataFrame, pair_name: str, top_n: int, *, largest: bool
) -> pd.DataFrame:
    src, tgt = parse_pair(pair_name)
    ca, cb = f"count_{src}", f"count_{tgt}"
    part = df.nlargest(top_n, "shift") if largest else df.nsmallest(top_n, "shift")
    out = part[["word", "shift", ca, cb]].copy()
    out.insert(0, "rank", range(1, top_n + 1))
    return out.rename(columns={ca: "count_a", cb: "count_b"})

## Top *N* highest distance (most shifted) per pair

In [3]:
from IPython.display import Markdown, display

for path, src, tgt in shift_files:
    df = load_ppmi_tsv(path, src, tgt)
    pair_name = df["pair"].iloc[0]
    display(
        Markdown(
            f"### `{pair_name}` — **count_a** = `{src}` (source), **count_b** = `{tgt}` (target)"
        )
    )
    display(pairwise_top_n(df, pair_name, TOP_N, largest=True))

## Top *N* lowest distance (most stable) per pair

In [4]:
from IPython.display import Markdown, display

for path, src, tgt in shift_files:
    df = load_ppmi_tsv(path, src, tgt)
    pair_name = df["pair"].iloc[0]
    display(
        Markdown(
            f"### `{pair_name}` — **count_a** = `{src}` (source), **count_b** = `{tgt}` (target)"
        )
    )
    display(pairwise_top_n(df, pair_name, TOP_N, largest=False))

## Words in all loaded pairwise vocabularies

Inner join on `word`, **mean_shift** = mean of per-pair distances.

In [5]:
if len(shift_files) >= 2:
    merged = None
    for path, a, b in shift_files:
        df = load_ppmi_tsv(path, a, b)
        pname = f"{a}_to_{b}"
        sub = df.rename(columns={"shift": f"shift_{pname}"})[["word", f"shift_{pname}"]]
        merged = sub if merged is None else merged.merge(sub, on="word", how="inner")
    shift_cols = [c for c in merged.columns if c.startswith("shift_")]
    merged["mean_shift"] = merged[shift_cols].mean(axis=1)
    for c in CORPORA:
        wv = get_wv(c)
        merged[f"count_{c}"] = [wv_word_count(wv, w) for w in merged["word"]]
    count_cols = [f"count_{c}" for c in CORPORA]
    print(f"Words in intersection of {len(shift_files)} PPMI pair files: {len(merged)}")
    show_cols = ["word", "mean_shift"] + shift_cols + count_cols
    display(merged.nlargest(TOP_N, "mean_shift")[show_cols])
    display(merged.nsmallest(TOP_N, "mean_shift")[show_cols])
else:
    print("Need at least two PPMI TSV files for merge.")

Need at least two PPMI TSV files for merge.


## Optional: summary stats per file

In [6]:
for path, src, tgt in shift_files:
    df = load_ppmi_tsv(path, src, tgt)
    print(f"{path.name}: n={len(df)}, shift min/median/max = {df['shift'].min():.4f} / {df['shift'].median():.4f} / {df['shift'].max():.4f}")